# בניית אפליקציות ליצירת תמונות (OpenAI)

יש ל-LLM יותר מיצירת טקסט. ניתן גם ליצור תמונות מתיאורי טקסט. תמונות כמדיום שימושיות בתחומי מדטק, אדריכלות, תיירות, פיתוח משחקים, שיווק ועוד. בשיעור זה נעבוד עם דגמי **GPT Image** העכשוויים בפלטפורמת OpenAI.

## מטרות הלמידה

בסוף שיעור זה תוכל/י:

- להסביר מהי יצירת תמונות ואיפה היא שימושית.
- להבין את משפחת הדגמים `gpt-image` וכיצד היא שונה מדגמי DALL·E הוותיקים.
- לבנות אפליקציה ליצירת תמונות ולערוך תמונות.

## מהי יצירת תמונות?

דגמי יצירת תמונות יוצרים תמונות מתוך הנחיית טקסט. דגמים מודרניים כמו `gpt-image` לומדים את הקשר בין טקסט לתמונה במהלך האימון, ואז הופכים באופן איטרטיבי רעש אקראי לתמונה התואמת את ההנחיה שלך.

שתי משפחות ידועות של דגמי תמונות הן:

- **`gpt-image` (OpenAI)** — הדור הנוכחי המשמש בשיעור זה. הוא תומך ביצירת תמונות מטקסט ועריכת תמונות (inpainting עם מסכה).
- **Midjourney** — דגם של צד שלישי פופולרי עם שירות משל עצמו וזרימת עבודה דרך דיסקורד.

> דגמי התמונות הישנים של OpenAI — **DALL·E 2** ו-**DALL·E 3** — הם וותיקים. DALL·E 3 כבר לא זמין לפריסות חדשות, ותכונות כמו `create_variation` היו קיימות רק ב-DALL·E 2. השתמש בדגמי `gpt-image` לאפליקציות חדשות.

> **חשוב:** דגמי `gpt-image` מחזירים את התמונה שנוצרה כ-**base64** (`b64_json`), לא ככתובת URL. הקוד שלך מפענח את מחרוזת ה-base64 לבייטים ושומר אותה — אין כתובת URL של תמונה להורדה.


## בניית אפליקציית יצירת תמונות ראשונה שלך

אז מה זה דורש לבנות אפליקציית יצירת תמונות? אתה צריך את הספריות הבאות:

- **python-dotenv**, מומלץ מאוד להשתמש בספריה זו כדי לשמור את הסודות שלך בקובץ *.env* בנפרד מהקוד.
- **openai**, ספריה זו היא זו שישתמשו בה כדי לתקשר עם OpenAI API.
- **pillow**, לעבודה עם תמונות בפייתון.


1. צור קובץ בשם *.env* עם התוכן הבא:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. אספו את הספריות לעיל בקובץ בשם *requirements.txt* כך:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. לאחר מכן, צרו סביבה וירטואלית והתקינו את הספריות:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> For Windows, use the following commands to create and activate your virtual environment:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Add the following code in file called *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # import dotenv
    dotenv.load_dotenv()

    # Create OpenAI object (reads OPENAI_API_KEY from your .env)
    client = openai.OpenAI()


    try:
        # Create an image by using the image generation API
        generation_response = client.images.generate(
            model="gpt-image-2",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
            size='1024x1024',
            n=1
        )
        # Set the directory for the stored image
        image_dir = os.path.join(os.curdir, 'images')

        # If the directory doesn't exist, create it
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Initialize the image path (note the filetype should be png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image models return the image as base64 (b64_json), not a URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Display the image in the default image viewer
        image = Image.open(image_path)
        image.show()

    # catch exceptions
    except openai.BadRequestError as err:
        print(err)

    ```

Let's explain this code:

- First, we import the libraries we need, including the OpenAI library, the dotenv library, the base64 module, and the Pillow library.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- After that, we create the client, which reads the API key from your ``.env``.

    ```python
    # Create OpenAI object
    client = openai.OpenAI()
    ```

- Next, we generate the image:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-2",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Enter your prompt text here
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` models return the image as a **base64** string in `data[0].b64_json`. We decode it to bytes and write it to a file — there's no URL to download.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Lastly, we open the image and use the standard image viewer to display it:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### More details on generating the image

Let's look at the parameters of `images.generate`:

- **model**, is the image model to use (for example `gpt-image-2`).
- **prompt**, is the text prompt used to generate the image. Here it's "Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils".
- **size**, is the size of the generated image (for example `1024x1024`, `1536x1024`, `1024x1536`, or `"auto"`).
- **n**, is the number of images generated. Here we generate one.

> Image models don't take a `temperature` parameter — that's a text-generation control. To get variety, call the API again; to reduce variety, make your prompt more specific.

## Additional capabilities of image generation

You've seen how to generate an image with a few lines of Python. `gpt-image` models can also **edit** an existing image. By providing an image, an optional **mask** (which marks the area to change), and a prompt, you can alter part of an image — for example, add a hat to our bunny.

```python
response = client.images.edit(
  model="gpt-image-2",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# edits are also returned as base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

The base image contains just the rabbit; the final image has the hat on the rabbit.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
